# 4.02 Web Crawl Loaders

웹 문서를 `Document`로 가져오는 네 가지 경로를 비교한다.

- **단일 URL 묶음**: `WebBaseLoader`
- **링크 트리 재귀**: `RecursiveUrlLoader`
- **사이트맵 기반**: `SitemapLoader`
- **관리형 크롤러 API**: `FireCrawlLoader` (Firecrawl 키 필요)

## 학습 목표

- `WebBaseLoader`의 `header_template` · `bs_kwargs` · `requests_per_second` 튜닝 포인트를 익힌다
- `RecursiveUrlLoader(max_depth=...)`로 한 도메인 내부 문서만 재귀 수집한다
- `SitemapLoader(filter_urls=...)`로 대량의 공식 문서에서 필요한 경로만 뽑는다
- `FireCrawlLoader`의 `mode="scrape" | "crawl" | "map"` 차이를 이해한다
- 수백 URL을 받을 때 `aload()` 비동기 메서드로 처리 시간을 줄인다

## 언제 쓰나

- 공식 문서 전체를 RAG 코퍼스로 구축 (LangChain 자체 문서, FastAPI 문서 등)
- 뉴스/블로그처럼 URL 리스트가 고정된 경우 → `WebBaseLoader`
- 구조가 복잡한 SaaS 사이트 → `FireCrawlLoader`가 JS 렌더링·robots.txt 처리 대신 해줌

## 4.02.1 환경 설정

필요 패키지: `langchain-community`, `beautifulsoup4`, `lxml`, `requests`. Firecrawl 섹션만 `firecrawl-py`와 `FIRECRAWL_API_KEY`가 추가로 필요.

probe는 실제 네트워크가 되는지 `requests.get`으로 확인하고, 안 되면 `nbconvert`가 즉시 멈춘다.

In [ ]:
# !pip install -U langchain-community beautifulsoup4 lxml requests firecrawl-py
import os, requests, nest_asyncio
nest_asyncio.apply()  # jupyter event loop 안에서 asyncio.run 허용
from dotenv import load_dotenv
load_dotenv(override=True)

PROBE_URL = "https://docs.langchain.com/oss/python/langchain/overview"
resp = requests.get(PROBE_URL, timeout=10, headers={"User-Agent": "ch-langchain-nb/1.0"})
assert resp.status_code == 200, f"probe 실패: {resp.status_code}"
print("probe ok:", resp.status_code, "|", len(resp.content), "bytes")

## 4.02.2 기본 로드 — `WebBaseLoader`

- 단일 URL: `WebBaseLoader("https://...")`
- 다수 URL: `WebBaseLoader(web_paths=[url1, url2, ...])`
- `bs_kwargs={"parse_only": SoupStrainer("main")}`로 본문 영역만 파싱해 잡음을 줄이는 패턴이 흔하다
- `requests_per_second=2` 기본값은 정중한 크롤링 속도

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from bs4 import SoupStrainer

loader = WebBaseLoader(
    web_paths=[PROBE_URL],
    header_template={"User-Agent": "ch-langchain-nb/1.0"},
    bs_kwargs={"parse_only": SoupStrainer(["article", "main", "div"])},
    requests_per_second=2,
    raise_for_status=True,
    show_progress=False,
)
docs = loader.load()
print("docs:", len(docs))
print("source:", docs[0].metadata.get("source"))
print("chars:", len(docs[0].page_content))
print("preview:", docs[0].page_content[:200].replace("\n", " "))

### 비동기 대량 로드 `aload()`

`WebBaseLoader`는 URL 리스트를 받으면 내부적으로 `asyncio.gather`로 병렬 fetch 한다 (`aload()` 호출 시).
직렬 `.load()` 대비 네트워크 대기 시간을 크게 줄인다.

In [ ]:
import time

URLS = [
    "https://docs.langchain.com/oss/python/langchain/overview",
    "https://docs.langchain.com/oss/python/langgraph/overview",
    "https://docs.langchain.com/oss/python/releases/v1",
]

# WebBaseLoader.load()는 여러 URL을 순차 GET 한다.
# aload()는 내부에서 aiohttp로 병렬 fetch — jupyter + Python 3.14에서는 nest_asyncio와 조합이 까다로워
# 프로덕션 스크립트에서는 그대로 쓰되, 노트북 실습에서는 load()로 단순화.
multi = WebBaseLoader(web_paths=URLS, requests_per_second=4, show_progress=False)

t0 = time.perf_counter()
batch = multi.load()
print(f"load: {len(batch)} docs in {(time.perf_counter() - t0)*1000:.0f} ms")
for d in batch:
    print("-", d.metadata["source"], "|", len(d.page_content), "chars")

# 참고: 비동기 버전 — 별도 Python 스크립트에서 실행
# >>> docs = await multi.aload()   # async def 안에서 사용


## 4.02.3 재귀 크롤러 — `RecursiveUrlLoader`

한 URL에서 시작해 같은 도메인의 링크를 **깊이 제한**으로 따라간다.

- `max_depth`: 시작 URL이 depth=1, 자식 링크가 depth=2 …
- `prevent_outside=True`: 다른 도메인 차단
- `extractor`: HTML → 텍스트 함수 주입 (기본 원문 HTML). BeautifulSoup로 **텍스트만** 뽑고 싶다면 아래처럼.
- `exclude_dirs`: `("/blog/", "/api/")` 같이 제외 경로 지정

In [ ]:
from langchain_community.document_loaders import RecursiveUrlLoader
from bs4 import BeautifulSoup

def html_to_text(html: str) -> str:
    return BeautifulSoup(html, "html.parser").get_text(" ", strip=True)

rec = RecursiveUrlLoader(
    url="https://docs.langchain.com/oss/python/releases/v1",
    max_depth=2,                 # 시작 페이지 + 한 단계 자식까지
    prevent_outside=True,
    use_async=False,
    timeout=10,
    extractor=html_to_text,
    headers={"User-Agent": "ch-langchain-nb/1.0"},
)
pages = rec.load()
print("수집 페이지:", len(pages))
for d in pages[:5]:
    print("-", d.metadata.get("source"), "|", len(d.page_content), "chars")

## 4.02.4 `SitemapLoader` — `filter_urls` 로 범위 좁히기

사이트맵(`sitemap.xml`)이 있는 프로젝트는 이걸 그대로 쓰는 게 가장 정확하다.

- `filter_urls`: 정규식 리스트. 지정한 패턴과 **매칭되는 URL만** 로드.
- `restrict_to_same_domain=True`(기본): 사이트맵 외부 링크 차단
- `continue_on_failure=True`: 일부 URL이 4xx여도 전체가 멈추지 않음
- `blocksize` / `blocknum`: 수만 URL짜리 사이트맵을 **샤딩**해서 병렬 로드

실습에서는 사이트맵 크기를 감안해 `filter_urls`로 5개 안팎만 뽑는다.

In [ ]:
# 네트워크 환경에 따라 사이트맵 페이로드가 매우 클 수 있어 filter_urls로 좁힌다.
# 예: 아래는 langchain.com 블로그 사이트맵에서 /v-1로 시작하는 포스트만 추림 (라이브 URL이 바뀌면 정규식을 조정).
#
# from langchain_community.document_loaders import SitemapLoader
# sm = SitemapLoader(
#     web_path="https://blog.langchain.com/sitemap-posts.xml",
#     filter_urls=[r".*langchain-v1.*"],
#     restrict_to_same_domain=True,
#     continue_on_failure=True,
# )
# sm_docs = sm.load()
# print('sitemap docs:', len(sm_docs))
# for d in sm_docs[:3]:
#     print('-', d.metadata.get('source'))

print("SitemapLoader 예시: 공식 사이트맵 URL이 프로젝트마다 다르므로 주석 참조. 아래 샘플은 offline 친화적으로 로컬 사이트맵을 파싱한다.")

from pathlib import Path
from langchain_community.document_loaders import SitemapLoader

local_sm = Path("./_sitemap.xml").resolve()
local_sm.write_text("""<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">
  <url><loc>https://docs.langchain.com/oss/python/langchain/overview</loc></url>
  <url><loc>https://docs.langchain.com/oss/python/langgraph/overview</loc></url>
</urlset>
""", encoding="utf-8")

sm = SitemapLoader(web_path=str(local_sm), is_local=True, continue_on_failure=True)
sm_docs = sm.load()
print("local sitemap docs:", len(sm_docs))
for d in sm_docs:
    print("-", d.metadata.get("source"), "|", len(d.page_content), "chars")

## 4.02.5 `FireCrawlLoader` — 관리형 크롤러 (키 필요)

Firecrawl은 JS 렌더링·robots.txt 준수·markdown 변환을 서버 측에서 처리한다. 3가지 주요 `mode`:

| mode | 동작 | 과금 단위 |
|------|------|-----------|
| `scrape` | 지정 URL 1개 → markdown/HTML/screenshots | 페이지당 |
| `crawl` | URL 루트에서 깊이 탐색 → 다수 페이지 | 페이지 수 |
| `map` | 사이트의 모든 링크 발견만 | 페이지당 |

`FIRECRAWL_API_KEY`가 없으면 key-gated 상태. 아래 셀은 키가 있을 때만 실행되고 없으면 즉시 assert로 멈춘다.

In [ ]:
# 키가 있을 때만 실행 — 없으면 아래 한 줄이 실패하며 뒤 셀로 넘어가지 않는다.
# assert os.environ.get("FIRECRAWL_API_KEY"), "FIRECRAWL_API_KEY 미설정"
#
# from langchain_community.document_loaders.firecrawl import FireCrawlLoader
# fc = FireCrawlLoader(
#     url="https://docs.langchain.com/oss/python/langchain/overview",
#     mode="scrape",
#     api_key=os.environ["FIRECRAWL_API_KEY"],
#     params={"formats": ["markdown"]},
# )
# firecrawl_docs = fc.load()
# print(firecrawl_docs[0].page_content[:300])

print("Firecrawl 섹션은 키가 있을 때 위 주석을 해제하고 실행하라.")
print("FIRECRAWL_API_KEY 설정 여부:", bool(os.environ.get("FIRECRAWL_API_KEY")))

## 4.02.6 split · embed 연동 — 한 줄 RAG

웹 문서는 보통 HTML 잡음이 많으므로 splitter를 거쳐 의미 단위로 자르는 게 유리하다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

assert os.environ.get("OPENAI_API_KEY"), "Chroma 임베딩용 OPENAI_API_KEY 필요"

chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)
store = Chroma.from_documents(chunks, embedding=OpenAIEmbeddings(model="text-embedding-3-small"), collection_name="web_demo")

for hit in store.similarity_search("What is LangChain?", k=2):
    print("-", hit.page_content[:150].replace("\n", " "))

## 체크리스트

- [ ] `User-Agent` 헤더를 커스터마이즈해 차단 방지
- [ ] `bs_kwargs` / `extractor`로 본문만 추출해 청크 품질 개선
- [ ] 대량 URL은 `aload()` 비동기 경로 사용
- [ ] 재귀 크롤은 `max_depth` + `prevent_outside`로 폭주 방지
- [ ] 사이트맵이 있으면 `SitemapLoader(filter_urls=...)`를 최우선으로
- [ ] JS가 많은 SaaS 사이트는 `FireCrawlLoader`가 편하다(유료)

## 다음

- `03_cloud_storage.ipynb` — S3/Azure Blob/GCS/Drive 로더
- `../06_text_splitters/` — HTML/Markdown 인식 splitter

## 참고

- WebBaseLoader 가이드: https://python.langchain.com/docs/how_to/document_loader_web/
- RecursiveUrlLoader: https://python.langchain.com/docs/integrations/document_loaders/recursive_url/
- Firecrawl: https://docs.firecrawl.dev/